# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [24]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [25]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [26]:
print(spaceship.shape)
print(spaceship.isnull().sum())

(8693, 14)
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [27]:
spaceship = spaceship.drop(columns=["PassengerId", "Name", "Cabin"])

In [28]:
spaceship = spaceship.dropna()
spaceship.shape

(6923, 11)

In [29]:
print(spaceship.columns.tolist())

['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported']


In [30]:
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5538, 10)
(1385, 10)
(5538,)
(1385,)


In [31]:
from sklearn.preprocessing import StandardScaler

features_to_scale = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
scaler = StandardScaler()
X_train[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])
print(X_train[features_to_scale].head())
print(X_test[features_to_scale].head())

           Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck
1869 -0.819055    -0.339190  -0.283271      0.090081  0.190558 -0.268247
6494  0.415547    -0.305748  -0.283271      1.468337 -0.276306 -0.242742
2852  1.444382    -0.339190   0.640345     -0.278936  1.274691 -0.271890
3131  1.650149    -0.336150  -0.272524      0.765871 -0.276306 -0.271890
32    0.209780    -0.339190   0.224210     -0.158894  0.103618  0.139822
           Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck
588   0.141191    -0.339190  -0.283271     -0.278936 -0.276306 -0.272801
4802 -0.887644    -0.273827  -0.283271     -0.278936  1.369455 -0.164408
4053 -0.064576    -0.339190   1.713818     -0.278936  1.586803 -0.248207
1658 -0.407521    -0.118780   0.009277     -0.253742 -0.233706 -0.271890
2389  0.895670    -0.336150   1.522169     -0.278936 -0.198930 -0.263692


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [32]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)

ValueError: could not convert string to float: 'Earth'

- Evaluate your model

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = gb_model.predict(X_test)
print("Accuracy base:", accuracy_score(y_test, y_pred))

Accuracy base: 0.8036101083032491


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5]
}

- Run Grid Search

In [ ]:
grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}


- Evaluate your model

In [ ]:
best_gb_model = grid_search.best_estimator_
y_pred_best = best_gb_model.predict(X_test)
print("Accuracy after tuning:", accuracy_score(y_test, y_pred_best))

Accuracy after tuning: 0.8101083032490974
